# Project2 - Host program

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from pynq import Overlay
import pynq.lib.dma
from pynq import Xlnk

ol = Overlay('phase.bit')

# Download your bitstream to FPGA
t_before_bitstream = time.time()
ol.download()
t_after_bitstream = time.time()

print(t_after_bitstream - t_before_bitstream, 'seconds to program bitstream')

In [ ]:
dmaIR = ol.axi_dma_0 # First DMA
dmaQT = ol.axi_dma_1 # Second DMA
ph_ip = ol.phasedetector_0 # Your IP

xlnk = Xlnk() # Contiguous Memory Allocator (CMA)

In [ ]:
length = 1024

# Open input/output files
fI = open('input_i.dat','r')
fQ = open('input_q.dat','r')
fG = open('out_gold.dat', 'r')

# Allocate regular numpy arrays to store input and output
inp_I = np.empty([length,], dtype=np.float32)
inp_Q = np.empty([length,], dtype=np.float32)
golden_R = np.empty([length,], dtype=np.float32)
golden_T = np.empty([length,], dtype=np.float32)

# Store data into arrays
for i in range (0, length):
    golden_R[i], golden_T[i] = [np.float32(x) for x in next(fG).split()]
    inp_I[i] = np.float32(next(fI))
    inp_Q[i] = np.float32(next(fQ))

In [ ]:
plt.plot(golden_R)

In [ ]:
plt.plot(golden_T)

In [ ]:
print("Golden thetas at the R peaks are:\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n".format(golden_T[31],golden_T[63],golden_T[95],golden_T[127],golden_T[159],golden_T[191],golden_T[223],golden_T[255],golden_T[287],golden_T[319],golden_T[351],golden_T[383],golden_T[415],golden_T[447],golden_T[479],golden_T[511]))

## Complete the following block

In [ ]:
# Allocate CMA arrays for DMA
in_bufferI  = xlnk.cma_array(shape=(length,), dtype=np.float32)
in_bufferQ  = xlnk.cma_array(shape=(length,), dtype=np.float32)
out_bufferR = xlnk.cma_array(shape=(length,), dtype=np.float32)
out_bufferT = xlnk.cma_array(shape=(length,), dtype=np.float32)

# Copy regular numpy arrays to CMA arrays
np.copyto(in_bufferI, inp_I)
np.copyto(in_bufferQ, inp_Q)

# Write length using MMIO (register offsets from xphasedetector_hw.h)
ph_ip.write(0x10, length)

t_start = time.time()
ph_ip.write(0x00, 0x01)                    # ap_start
dmaIR.recvchannel.transfer(out_bufferR)
dmaQT.recvchannel.transfer(out_bufferT)
dmaIR.sendchannel.transfer(in_bufferI)
dmaQT.sendchannel.transfer(in_bufferQ)
dmaIR.sendchannel.wait()
dmaQT.sendchannel.wait()
dmaIR.recvchannel.wait()
dmaQT.recvchannel.wait()
t_stop = time.time()

in_bufferI.close()
in_bufferQ.close()
print(t_stop - t_start, 'seconds to execute on hardware')


In [ ]:
plt.plot(out_bufferR)

In [ ]:
plt.plot(out_bufferT)

In [ ]:
print("Output thetas at the R peaks are:\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n {}\n".format(out_bufferT[31],out_bufferT[63],out_bufferT[95],out_bufferT[127],out_bufferT[159],out_bufferT[191],out_bufferT[223],out_bufferT[255],out_bufferT[287],out_bufferT[319],out_bufferT[351],out_bufferT[383],out_bufferT[415],out_bufferT[447],out_bufferT[479],out_bufferT[511]))